In [12]:
# Cell 1: Import libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchsummary import summary


In [13]:
# Cell 2: Set random seeds
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Random seed set to {SEED}")


Random seed set to 42


In [14]:
# Cell 3: Download dataset
import os
import urllib.request
import zipfile

data_url = 'https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip'
data_path = 'data.zip'

if not os.path.exists('data'):
    print(f"Downloading dataset...")
    urllib.request.urlretrieve(data_url, data_path)
    print(f"Extracting...")
    with zipfile.ZipFile(data_path, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Dataset ready!")
else:
    print("Dataset already exists.")


Dataset already exists.


In [15]:
# Cell 4: Build CNN Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class HairCNN(nn.Module):
    def __init__(self):
        super(HairCNN, self).__init__()
        # Layer 1: Look for patterns in image (32 filters, 3x3 size)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=0, stride=1)
        
        # Layer 2: Shrink image size (2x2 pooling)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Layer 3: Combine features (64 neurons)
        self.fc1 = nn.Linear(32 * 99 * 99, 64)
        
        # Layer 4: Final decision (1 output: straight=0, curly=1)
        self.fc2 = nn.Linear(64, 1)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))  # Detect patterns
        x = self.pool(x)                # Shrink
        x = x.view(x.size(0), -1)      # Flatten to 1D
        x = torch.relu(self.fc1(x))    # Combine features
        x = self.fc2(x)                 # Final score
        return x

model = HairCNN().to(device)
print(model)


Using device: cpu
HairCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=313632, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


In [5]:
# Question 1 - Loss Function
criterion = nn.BCEWithLogitsLoss()


In [17]:
# Cell 6: Question 2 - Count Parameters
# Method 1: Using torchsummary
summary(model, input_size=(3, 200, 200))

# Method 2: Manual count
total_params = sum(p.numel() for p in model.parameters())
print(f"Q2 Answer: {total_params:,} parameters")


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
         MaxPool2d-2           [-1, 32, 99, 99]               0
            Linear-3                   [-1, 64]      20,072,512
            Linear-4                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 11.96
Params size (MB): 76.57
Estimated Total Size (MB): 89.00
----------------------------------------------------------------
Q2 Answer: 20,073,473 parameters


In [18]:
    # Transform images: resize, convert to tensor, normalize
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

# Load images from folders
train_dataset = datasets.ImageFolder('data/train', transform=train_transforms)
test_dataset = datasets.ImageFolder('data/test', transform=test_transforms)

# Create batches of 20 images
train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=20, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Test images: {len(test_dataset)}")


Training images: 800
Test images: 201


In [19]:
# Setup data loaders and optimizer first
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim

# Define transforms
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder('data/train', transform=train_transforms)
validation_dataset = datasets.ImageFolder('data/test', transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=20, shuffle=False)

# Setup optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

# Training loop for 10 epochs
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)
    
    # Validation
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)


In [20]:
# Cell 8: Setup optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)
print("Optimizer ready!")


Optimizer ready!


In [22]:
# Cell 9: Training Loop (10 epochs)
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

print("Starting training...")
for epoch in range(num_epochs):
    # TRAINING
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()           # Reset gradients
        outputs = model(images)         # Predict
        loss = criterion(outputs, labels)  # Calculate error
        loss.backward()                 # Learn from error
        optimizer.step()                # Update weights
        
        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / len(train_dataset)
    train_acc = correct / total
    history['loss'].append(train_loss)
    history['acc'].append(train_acc)
    
    # VALIDATION
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_loss = val_loss / len(test_dataset)
    val_acc = val_correct / val_total
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print(" Training complete!")


Starting training...
Epoch 1/10 - Loss: 0.0032, Acc: 1.0000, Val Loss: 1.1329, Val Acc: 0.7164
Epoch 2/10 - Loss: 0.0028, Acc: 1.0000, Val Loss: 1.1689, Val Acc: 0.7114
Epoch 3/10 - Loss: 0.0026, Acc: 1.0000, Val Loss: 1.2083, Val Acc: 0.7114
Epoch 4/10 - Loss: 0.0023, Acc: 1.0000, Val Loss: 1.2108, Val Acc: 0.7114
Epoch 5/10 - Loss: 0.0022, Acc: 1.0000, Val Loss: 1.2111, Val Acc: 0.7114
Epoch 6/10 - Loss: 0.0020, Acc: 1.0000, Val Loss: 1.2171, Val Acc: 0.7114
Epoch 7/10 - Loss: 0.0018, Acc: 1.0000, Val Loss: 1.2453, Val Acc: 0.7114
Epoch 8/10 - Loss: 0.0017, Acc: 1.0000, Val Loss: 1.2411, Val Acc: 0.7114
Epoch 9/10 - Loss: 0.0016, Acc: 1.0000, Val Loss: 1.2478, Val Acc: 0.7114
Epoch 10/10 - Loss: 0.0015, Acc: 1.0000, Val Loss: 1.2759, Val Acc: 0.7114
 Training complete!


In [24]:
# Question 3
median_acc = np.median(history['acc'])
print(f" Q3 Answer: {median_acc:.2f}")


 Q3 Answer: 1.00


In [25]:
 #Question 4
std_loss = np.std(history['loss'])
print(f" Q4 Answer: {std_loss:.3f}")


 Q4 Answer: 0.001


In [28]:
 #Add augmentation
train_transforms_aug = transforms.Compose([
    transforms.RandomRotation(50),           
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)), 
    transforms.RandomHorizontalFlip(),       
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Reload training data with augmentation
train_dataset_aug = datasets.ImageFolder('data/train', transform=train_transforms_aug)
train_loader_aug = DataLoader(train_dataset_aug, batch_size=20, shuffle=True)

print(" Data augmentation ready!")


 Data augmentation ready!


In [29]:
print("Starting training with augmentation...")
for epoch in range(10):
    # Same training code as before but use train_loader_aug
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader_aug:  # Using augmented data
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / len(train_dataset_aug)
    train_acc = correct / total
    history['loss'].append(train_loss)
    history['acc'].append(train_acc)
    
    # Validation (same as before)
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_loss = val_loss / len(test_dataset)
    val_acc = val_correct / val_total
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+11}/{20} - "
          f"Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Starting training with augmentation...
Epoch 11/20 - Loss: 0.9886, Acc: 0.5600, Val Loss: 0.6755, Val Acc: 0.5920
Epoch 12/20 - Loss: 0.6342, Acc: 0.5750, Val Loss: 0.7232, Val Acc: 0.6766
Epoch 13/20 - Loss: 0.6000, Acc: 0.6887, Val Loss: 0.7870, Val Acc: 0.6418
Epoch 14/20 - Loss: 0.5654, Acc: 0.7063, Val Loss: 0.6404, Val Acc: 0.6965
Epoch 15/20 - Loss: 0.5654, Acc: 0.6925, Val Loss: 0.6256, Val Acc: 0.6318
Epoch 16/20 - Loss: 0.5328, Acc: 0.7262, Val Loss: 0.6278, Val Acc: 0.6965
Epoch 17/20 - Loss: 0.5209, Acc: 0.7438, Val Loss: 0.6513, Val Acc: 0.6915
Epoch 18/20 - Loss: 0.5166, Acc: 0.7225, Val Loss: 0.7714, Val Acc: 0.6020
Epoch 19/20 - Loss: 0.5316, Acc: 0.7113, Val Loss: 0.6465, Val Acc: 0.6965
Epoch 20/20 - Loss: 0.5111, Acc: 0.7300, Val Loss: 0.6253, Val Acc: 0.7015


In [30]:
# Question 5
# Get last 10 epochs (epochs 11-20, indices 10-19)
last_10_val_loss = history['val_loss'][10:]
mean_val_loss = np.mean(last_10_val_loss)
print(f" Q5 Answer: {mean_val_loss:.2f}")


 Q5 Answer: 0.68


In [31]:
# Question 6
# Get last 5 epochs (epochs 16-20, indices 15-19)
last_5_val_acc = history['val_acc'][15:]
avg_val_acc = np.mean(last_5_val_acc)
print(f" Q6 Answer: {avg_val_acc:.2f}")


 Q6 Answer: 0.68
